# Run Inference with LLaVA 1.5

Companion notebook for the **Multimodal LLMs** lesson on Codemia.

**What you will build**: a working vision-language inference setup. You will load LLaVA 1.5 7B in
4-bit quantization, caption a few photos, and then probe its vision-language alignment with
questions that have exact answers: how many objects, and where they are. Finally you will ask the
same questions with the image removed, which isolates what the vision encoder contributes.

## Setup

LLaVA 1.5 7B in 4-bit needs a CUDA GPU with about 6 GB free. A free Colab T4 is enough
(Runtime > Change runtime type > T4 GPU). `bitsandbytes`, which does the 4-bit quantization,
only runs on CUDA.

Without a GPU, the notebook loads `llava-interleave-qwen-0.5b`, a 0.5B model from the same LLaVA
family, in full precision on CPU (or Apple silicon). Every cell runs the same way; expect weaker
answers, which makes some of the probes more interesting rather than less.

In [ ]:
%pip install -q transformers accelerate bitsandbytes pillow matplotlib

In [ ]:
import io
import random
import re
import urllib.request

import matplotlib.pyplot as plt
import torch
from PIL import Image, ImageDraw
from transformers import AutoProcessor, BitsAndBytesConfig, LlavaForConditionalGeneration

HAS_CUDA = torch.cuda.is_available()
MODEL_ID = 'llava-hf/llava-1.5-7b-hf' if HAS_CUDA else 'llava-hf/llava-interleave-qwen-0.5b-hf'
random.seed(0)
print('model:', MODEL_ID)

## Concept recap

LLaVA is three parts glued together: a **vision encoder** (CLIP ViT-L/14 at 336 px) that turns
the image into a grid of patch embeddings, a small **projector** that maps each patch embedding
into the language model's token space, and a **language model** (Vicuna 7B) that reads those
projected patches as if they were words. A 336 px image becomes a 24 x 24 grid, so 576 "image
tokens" are placed into the prompt where `<image>` appears.

Nothing in that design counts or measures. Whatever the model knows about "how many" and "where"
has to survive the patch grid, the projection and the language model's own habits. The probes
below measure how much does.

## Load the model

4-bit NF4 quantization stores each weight in 4 bits with a per-block scale, and computes in fp16.
The 7B model drops from about 14 GB in fp16 to under 4 GB, which is what makes it fit on a T4.

In [ ]:
if HAS_CUDA:
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                               bnb_4bit_compute_dtype=torch.float16)
    model = LlavaForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=quant, device_map='auto')
else:
    device = 'mps' if torch.backends.mps.is_available() else 'cpu'
    model = LlavaForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to(device)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print(f'memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB')

## One function for every question

`ask(question, image)` builds the chat prompt with the processor's own template, so the image
placeholder lands where this model expects it. Pass `image=None` to ask the identical question
with no image at all. Decoding is greedy, so every answer is reproducible.

In [ ]:
def ask(question, image=None, max_new_tokens=60):
    content = ([{'type': 'image'}] if image is not None else []) + [{'type': 'text', 'text': question}]
    prompt = processor.apply_chat_template([{'role': 'user', 'content': content}],
                                           add_generation_prompt=True)
    inputs = processor(images=image, text=prompt, return_tensors='pt').to(model.device)
    if 'pixel_values' in inputs:
        inputs['pixel_values'] = inputs['pixel_values'].to(model.dtype)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()


def load(url):
    with urllib.request.urlopen(url) as r:
        return Image.open(io.BytesIO(r.read())).convert('RGB')

How many tokens does one image cost? Count the image placeholder tokens the processor inserted.
Expect 576 for LLaVA 1.5 (a 24 x 24 grid). The CPU fallback uses a SigLIP encoder at 384 px with
14 px patches, so it spends 27 x 27 = 729.

In [ ]:
COCO = 'http://images.cocodataset.org/val2017/'
cats = load(COCO + '000000039769.jpg')
prompt = processor.apply_chat_template(
    [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': 'hi'}]}],
    add_generation_prompt=True)
ids = processor(images=cats, text=prompt, return_tensors='pt')['input_ids'][0]
image_token = model.config.image_token_index
print('prompt tokens:', len(ids), '| image tokens:', int((ids == image_token).sum()))

## Experiment 1: captions

Four photos from the COCO validation set. Look at each one and judge the caption yourself before
reading the next section.

In [ ]:
PHOTOS = {
    'cats': COCO + '000000039769.jpg',
    'skier': COCO + '000000000785.jpg',
    'bedroom': COCO + '000000000632.jpg',
    'baseball': COCO + '000000002153.jpg',
}
images = {k: load(u) for k, u in PHOTOS.items()}

fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
for ax, (k, im) in zip(axes, images.items()):
    ax.imshow(im); ax.set_title(k); ax.axis('off')
plt.show()

for k, im in images.items():
    print(f'{k:<9}', ask('Describe this image in one sentence.', im))

## Experiment 2: count and locate, with and without the image

Each question has an answer checked by eye against the photo. The same question is asked twice:
once with the image, once with no image. The no-image run uses the same language model, so any
answer it gets right came from priors ("cats on a couch usually come in ones or twos"), not from
seeing. The difference between the two columns is what the vision encoder adds.

In [ ]:
PROBES = [
    ('cats', 'How many cats are in the image? Answer with a number.', '2'),
    ('cats', 'How many remote controls are in the image? Answer with a number.', '2'),
    ('skier', 'What color is the skier\'s jacket? Answer with one word.', 'red'),
    ('bedroom', 'Is the bookshelf on the left or the right side of the image? Answer left or right.', 'right'),
    ('baseball', 'How many people are visible in the image? Answer with a number.', '4'),
]
NUMBER_WORDS = ('zero one two three four five six seven eight nine ten eleven twelve thirteen '
                'fourteen fifteen sixteen seventeen eighteen nineteen twenty').split()


def parse_count(answer):
    # Models answer "2", "Two" or "There are two cats". Take the first number in any form.
    m = re.search(r'\b(\d+|' + '|'.join(NUMBER_WORDS) + r')\b', answer.lower())
    if not m:
        return None
    w = m.group(1)
    return int(w) if w.isdigit() else NUMBER_WORDS.index(w)


def correct(answer, truth):
    if truth.isdigit():
        return parse_count(answer) == int(truth)
    return truth in answer.lower()


rows = []
for name, q, truth in PROBES:
    seen, blind = ask(q, images[name], 12), ask(q, None, 12)
    rows.append((correct(seen, truth), correct(blind, truth)))
    print(f'{name:<9} truth={truth:<6} with image: {seen[:28]:<30} no image: {blind[:40]}')
print(f'\ncorrect with image: {sum(r[0] for r in rows)}/{len(rows)}   '
      f'without: {sum(r[1] for r in rows)}/{len(rows)}')

## Variations

Real photos give five data points. Synthetic images give as many as you like, with answers that
are exact by construction.

**Counting.** Draw 1 to 10 dots on a blank canvas and ask how many there are.

In [ ]:
def dots(n, size=336, r=14):
    im = Image.new('RGB', (size, size), 'white')
    d, placed = ImageDraw.Draw(im), []
    while len(placed) < n:  # rejection sampling keeps dots from touching
        x, y = random.randint(r, size - r), random.randint(r, size - r)
        if all((x - a) ** 2 + (y - b) ** 2 > (3 * r) ** 2 for a, b in placed):
            placed.append((x, y))
            d.ellipse([x - r, y - r, x + r, y + r], fill='black')
    return im


TRIALS = 3
acc = {}
for n in range(1, 11):
    hits = 0
    for _ in range(TRIALS):
        a = ask('How many black dots are in the image? Answer with a number.', dots(n), 6)
        hits += parse_count(a) == n
    acc[n] = hits / TRIALS
print({n: f'{v:.0%}' for n, v in acc.items()})

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.plot(list(acc), list(acc.values()), 'o-')
plt.xlabel('dots in the image'); plt.ylabel('exact count accuracy')
plt.title(f'Counting, {TRIALS} images per point'); plt.ylim(-0.05, 1.05); plt.grid(alpha=0.3)
plt.show()

**Locating.** Put one red square in one of four quadrants and ask where it is. Random guessing
scores 25%.

In [ ]:
QUADS = {'top left': (0, 0), 'top right': (1, 0), 'bottom left': (0, 1), 'bottom right': (1, 1)}


def square_in(quadrant, size=336, s=60):
    qx, qy = QUADS[quadrant]
    x = qx * size // 2 + random.randint(10, size // 2 - s - 10)
    y = qy * size // 2 + random.randint(10, size // 2 - s - 10)
    im = Image.new('RGB', (size, size), 'white')
    ImageDraw.Draw(im).rectangle([x, y, x + s, y + s], fill='red')
    return im


hits, total = 0, 0
for quadrant in QUADS:
    for _ in range(3):
        a = ask('Where is the red square? Answer with one of: top left, top right, '
                'bottom left, bottom right.', square_in(quadrant), 8).lower()
        hits += quadrant in a
        total += 1
print(f'quadrant accuracy: {hits}/{total} ({hits / total:.0%}, chance is 25%)')

Things worth noticing:

1. **Captions are the easy part.** Captioning is close to what LLaVA was trained on, and
   captions are forgiving: a sentence can be fluent and mostly right while getting a count
   wrong. Exact-answer probes are where alignment gets measured.
2. **The no-image column is not zero.** A language model guesses plausibly. Questions like
   "how many cats" have strong priors, so a correct answer with the image is only evidence of
   seeing when the blind answer was wrong.
3. **Counting falls off with the number of objects.** Nothing in a 24 x 24 patch grid represents
   a count, and once several objects share patches, the language model is estimating. Where the
   curve drops for your model is the practical limit for any feature that depends on counting.

## What to try next

- Repeat the dot experiment at a different image size. The processor resizes everything to the
  encoder's resolution, so small dots on a big canvas may vanish entirely.
- Ask for coordinates instead of quadrants ("give the bounding box as x1, y1, x2, y2 in 0-1
  units") and measure the error. LLaVA 1.5 was trained with some region data; see how precise it is.
- Swap in a newer open model (Qwen2-VL, LLaVA-OneVision) with the same `ask` function and compare
  the counting curves. Models that tile the image at higher resolution usually count further.